In [ ]:
!pip install torchdiffeq -q

In [ ]:
import pandas as pd
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from torchdiffeq import odeint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

SEQ_LEN    = 20
STRIDE     = 5
BATCH_SIZE = 32
LR         = 1e-3
NUM_EPOCHS = 300

DATA_DIR = Path('/content')  # папка, где лежат csv

csv_files = list(DATA_DIR.glob('*.csv'))

dfs = []

for file in csv_files:
    temp = pd.read_csv(file)

    # Если в csv нет колонки surface/terrain,
    # берем название поверхности из имени файла
    if 'surface' not in temp.columns:
        if 'terrain' in temp.columns:
            temp = temp.rename(columns={'terrain': 'surface'})
        else:
            temp['surface'] = file.stem

    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

def time_split(df, surface, val_ratio=0.2):
    sub = df[df['surface'] == surface].reset_index(drop=True)
    n = len(sub)
    cut = int(n * (1 - val_ratio))
    return sub.iloc[:cut], sub.iloc[cut:]

splits = {}

for surface in df['surface'].unique():
    train, val = time_split(df, surface, val_ratio=0.2)
    splits[surface] = {
        'train': train,
        'val': val
    }

print(f"loaded csv files: {len(csv_files)}")
print(f"surfaces: {list(splits.keys())}")

for surface, data in splits.items():
    print(f"{surface}: train: {len(data['train'])}, val: {len(data['val'])}")

loaded csv files: 4
surfaces: ['wet_asphalt', 'dry_concrete', 'ice', 'mud']
wet_asphalt: train: 23888, val: 5972
dry_concrete: train: 23883, val: 5971
ice: train: 23876, val: 5969
mud: train: 23881, val: 5971


In [ ]:
class TerrainDataset(Dataset):
    def __init__(self, sub_df, seq_len, stride):
        sub = sub_df.reset_index(drop=True)

        self.inputs  = torch.tensor(sub[['cmd_vx', 'cmd_wz']].values,  dtype=torch.float32)
        self.outputs = torch.tensor(sub[['odom_vx', 'odom_wz']].values, dtype=torch.float32)
        self.dts     = torch.tensor(sub['dt'].values,                  dtype=torch.float32)

        self.seq_len = seq_len
        self.indices = list(range(0, len(sub) - seq_len, stride))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        s = self.seq_len

        return (
            self.inputs[i:i+s],
            self.outputs[i:i+s],
            self.dts[i:i+s]
        )

SEQ_LEN    = 5
STRIDE     = 20
BATCH_SIZE = 128
LR         = 1e-3
NUM_EPOCHS = 30

surfaces = list(splits.keys())

train_loaders = {}
val_loaders = {}

for surface in surfaces:
    train_loaders[surface] = DataLoader(
        TerrainDataset(splits[surface]['train'], SEQ_LEN, STRIDE),
        batch_size=BATCH_SIZE,
        shuffle=True,
        pin_memory=torch.cuda.is_available()
    )

    val_loaders[surface] = DataLoader(
        TerrainDataset(splits[surface]['val'], SEQ_LEN, STRIDE),
        batch_size=BATCH_SIZE,
        shuffle=False,
        pin_memory=torch.cuda.is_available()
    )

print("Surfaces:", surfaces)

Surfaces: ['wet_asphalt', 'dry_concrete', 'ice', 'mud']


In [ ]:
class ODEFunc(torch.nn.Module):
    def __init__(self, hidden=16):
        super().__init__()

        self.net = torch.nn.Sequential(
            torch.nn.Linear(4, hidden),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, 2)
        )

    def forward(self, t, y):
        return self.net(y)

models = {}
opts = {}
schedulers = {}

for surface in surfaces:
    models[surface] = ODEFunc(hidden=16).to(device)

    opts[surface] = torch.optim.Adam(
        models[surface].parameters(),
        lr=LR
    )

    schedulers[surface] = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opts[surface],
        patience=5,
        factor=0.5
    )

In [ ]:
def run_epoch(model, loader, opt=None):
    training = opt is not None

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0

    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for cmds, states, dts in loader:
            cmds   = cmds.to(device)
            states = states.to(device)
            dts    = dts.to(device)

            batch_size, seq_len, _ = states.shape

            # Один общий t_span для всего batch.
            # Это быстрее, чем отдельный t_span для каждого элемента.
            mean_dt = dts[:, :-1].mean(dim=0)

            t_span = torch.cat([
                torch.zeros(1, device=device),
                torch.cumsum(mean_dt, dim=0)
            ])

            y0 = states[:, 0, :]  # (B, 2)

            def ode_fn(t, y):
                idx = torch.searchsorted(
                    t_span.contiguous(),
                    t.reshape(1)
                ).clamp(0, seq_len - 1).item()

                ctrl = cmds[:, idx, :]  # (B, 2)

                model_input = torch.cat([y, ctrl], dim=-1)  # (B, 4)

                return model.net(model_input)

            pred = odeint(
                ode_fn,
                y0,
                t_span,
                method='euler'
            )

            # pred: (T, B, 2) -> (B, T, 2)
            pred = pred.permute(1, 0, 2)

            loss = torch.nn.functional.mse_loss(pred, states)

            if training:
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
history = {'epoch': []}

for surface in surfaces:
    history[f'train_{surface}'] = []
    history[f'val_{surface}'] = []

for epoch in range(NUM_EPOCHS):
    history['epoch'].append(epoch)

    log_parts = [f"Epoch {epoch:3d}"]

    for surface in surfaces:
        train_loss = run_epoch(
            models[surface],
            train_loaders[surface],
            opts[surface]
        )

        val_loss = run_epoch(
            models[surface],
            val_loaders[surface]
        )

        schedulers[surface].step(val_loss)

        history[f'train_{surface}'].append(train_loss)
        history[f'val_{surface}'].append(val_loss)

        log_parts.append(
            f"{surface} train {train_loss:.6f} val {val_loss:.6f}"
        )

    print(" | ".join(log_parts))

Epoch   0 | wet_asphalt train 0.000108 val 0.000080 | dry_concrete train 0.000216 val 0.000106 | ice train 0.000192 val 0.000147 | mud train 0.000175 val 0.000154
Epoch   1 | wet_asphalt train 0.000072 val 0.000058 | dry_concrete train 0.000184 val 0.000076 | ice train 0.000131 val 0.000096 | mud train 0.000115 val 0.000099
Epoch   2 | wet_asphalt train 0.000054 val 0.000050 | dry_concrete train 0.000160 val 0.000059 | ice train 0.000088 val 0.000063 | mud train 0.000071 val 0.000060
Epoch   3 | wet_asphalt train 0.000048 val 0.000048 | dry_concrete train 0.000144 val 0.000050 | ice train 0.000062 val 0.000043 | mud train 0.000044 val 0.000038
Epoch   4 | wet_asphalt train 0.000045 val 0.000047 | dry_concrete train 0.000138 val 0.000046 | ice train 0.000046 val 0.000032 | mud train 0.000029 val 0.000027
Epoch   5 | wet_asphalt train 0.000043 val 0.000046 | dry_concrete train 0.000130 val 0.000044 | ice train 0.000037 val 0.000027 | mud train 0.000022 val 0.000022
Epoch   6 | wet_asphal

In [ ]:
import re
import pandas as pd
import torch
from google.colab import files

SAVE_DIR = '/content'

def safe_name(name):
    return re.sub(r'[^a-zA-Z0-9_]+', '_', name)

for surface in surfaces:
    name = safe_name(surface)

    history_surface = pd.DataFrame({
        'epoch': history['epoch'],
        'train_loss': history[f'train_{surface}'],
        'val_loss': history[f'val_{surface}']
    })

    history_path = f'{SAVE_DIR}/training_history_{name}.csv'
    model_path   = f'{SAVE_DIR}/neural_ode_{name}.pth'

    history_surface.to_csv(history_path, index=False)

    torch.save({
        'epoch': NUM_EPOCHS,
        'surface': surface,
        'model_state_dict': models[surface].state_dict(),
    }, model_path)

    files.download(history_path)
    files.download(model_path)

    print(f"Saved {surface}:")
    print(f"  {history_path}")
    print(f"  {model_path}")

print("Saved all surfaces!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved wet_asphalt:
  /content/training_history_wet_asphalt.csv
  /content/neural_ode_wet_asphalt.pth


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved mud:
  /content/training_history_mud.csv
  /content/neural_ode_mud.pth
Saved all surfaces!
